In [1]:
import sys

keys = []
for k in sys.modules.keys():
    if "SYXNAT" in k:
        # print(k)
        keys.append(k)
for k in keys:
    del sys.modules[k]

from SYXNAT.batch.upload_subjects import UploadSubjects
from SYXNAT.utils.utils import get_session
from SYXNAT.utils.subjects import delete_subject, create_subject
from SYXNAT.utils.scans import create_scan, get_scan
from SYXNAT.utils.subjects import create_subject
from SYXNAT.utils.resources import upload_resources
from SYXNAT.utils.experiments import create_experiment, update_experiment
from SYXNAT.utils.interfaces import MyScan, ScanType, ScanQuality, MySubject, MyExperiment, ExperimentType
import requests
from pydicom import dcmread

In [8]:
from pathlib import Path

In [12]:
dcm_file = dcmread('/Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RS_RTSTRUCT/Series7_1/RS.2025-05-15-015.SS1.dcm')

In [17]:
session = get_session()

In [15]:
# ── Step 1：刷新 SOP UID 缓存 ──────────────────────
print("刷新 sdcache ...")
resp = session.post(
    f"https://xnat.pemed.cn:9443/xapi/roi/projects/SYOncologySR/sdcache/SYZLXNAT_E00127",
    params={"cmd": "REFRESH"}
)
print(f"REFRESH 结果：{resp.status_code}  {resp.text}")

# # ── Step 2：验证缓存是否已更新 ─────────────────────
resp = session.get(
    f"https://xnat.pemed.cn:9443/xapi/roi/projects/SYOncologySR"
    f"/sessions/SYZLXNAT_E00127/scans/CT_Series7_183/uids"
)
sop_uids = resp.json()
print(f"缓存刷新后 SOP UID 数量：{len(sop_uids)}")

刷新 sdcache ...
REFRESH 结果：200  Cache refreshed for session SYZLXNAT_E00127
缓存刷新后 SOP UID 数量：0


In [16]:
print("【诊断1】查询 XNAT 核心文件列表")
resp = session.get(
    f"https://xnat.pemed.cn:9443/data/projects/SYOncologySR/experiments/SYZLXNAT_E00127"
    f"/scans/CT_Series7_183/resources/DICOM/files",
    params={"format": "json"}
)
result = resp.json()
result

【诊断1】查询 XNAT 核心文件列表


{'items': [{'children': [{'field': 'fields/field',
     'items': [{'children': [],
       'meta': {'create_event_id': 21951,
        'xsi:type': 'xnat:experimentData_field',
        'isHistory': False,
        'start_date': 'Fri Apr 24 15:03:50 UTC 2026'},
       'data_fields': {'field': '1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841',
        'xnat_experimentData_field_id': 238,
        'name': 'studyid'}},
      {'children': [],
       'meta': {'create_event_id': 21951,
        'xsi:type': 'xnat:experimentData_field',
        'isHistory': False,
        'start_date': 'Fri Apr 24 15:03:50 UTC 2026'},
       'data_fields': {'field': 'CBCT_CT_RTDOSE_REG_RTIMAGE_RTPLAN_RTSTRUCT_RTRECORD',
        'xnat_experimentData_field_id': 239,
        'name': 'modalities'}}]},
    {'field': 'scans/scan',
     'items': [{'children': [{'field': 'file',
         'items': [{'children': [],
           'meta': {'create_event_id': 22003,
            'xsi:type': 'xnat:resourceCatalog',
          

In [7]:
params = {
    "type": "RTSTRUCT",
    "overwrite": "false"
}

In [18]:
resp = session.put(
    'https://xnat.pemed.cn:9443/xapi/roi/projects/20260318/sessions/SYZLXNAT_E00004/collections/RS_PatientA',
    params=params,
    headers={"Content-Type": "application/octet-stream"},
    data=Path('/Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RS_RTSTRUCT/Series7_1/RS.2025-05-15-015.SS1.dcm').read_bytes()  # 关键：直接传二进制内容，不要用 files=
)

In [19]:
resp.status_code

200

In [11]:
resp.text

'Missing dependencies: SOP instance UID'

In [4]:
us = UploadSubjects(session, 'SYOncologySR', r'/Users/kukudehui/Desktop/XNAT_DATA/sorted_data')

In [5]:
us.creat_tasks()

In [6]:
us.handle_tasks_sequentially(
    subjects=True,
    experiments=True,
    scans=True,
    resources=True,
    # recover_index=841,
)

[SUCCESS] [Subject] 2025-05-15-015
[SUCCESS] [Experiment] 2025-05-15-015 study_1_2025-05-15-015
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series1_64
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series2_64
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series3_64
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series4_64
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series5_64
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series6_64
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 CT_Series7_183
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 RD_Series1_1
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 RE_Series1_1
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 RE_Series2_1
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 RE_Series3_1
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 RE_Series4_1
[SUCCESS] [Scan] 2025-05-15-015 study_1_2025-05-15-015 RE_Series5_1


✓ 上传完成: 100%|██████████| 18.1M/18.1M [00:21<00:00, 871kB/s]



全部 64 个文件上传成功!
[SUCCESS] [Resources] idx=0 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series1_64 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/CT_CT/Series1_64
[INFO] [Resources] idx=1 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series2_64 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/CT_CT/Series2_64


✓ 上传完成: 100%|██████████| 18.1M/18.1M [00:21<00:00, 876kB/s]



全部 64 个文件上传成功!
[SUCCESS] [Resources] idx=1 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series2_64 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/CT_CT/Series2_64
[INFO] [Resources] idx=2 2025-05-15-015 study_1_2025-05-15-015 CBCT_Series3_64 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/CT_CT/Series3_64


...


全部 2 个文件上传成功!
[SUCCESS] [Resources] idx=48 2025-05-15-015 study_1_2025-05-15-015 RT_Series7_2 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RT_RTRECORD/Series7_2
[INFO] [Resources] idx=49 2025-05-15-015 study_1_2025-05-15-015 RT_Series8_2 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RT_RTRECORD/Series8_2


✓ 上传完成: 100%|██████████| 4.22k/4.22k [00:03<00:00, 1.35kB/s]



全部 2 个文件上传成功!
[SUCCESS] [Resources] idx=49 2025-05-15-015 study_1_2025-05-15-015 RT_Series8_2 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RT_RTRECORD/Series8_2
[INFO] [Resources] idx=50 2025-05-15-015 study_1_2025-05-15-015 RT_Series9_2 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RT_RTRECORD/Series9_2


✓ 上传完成: 100%|██████████| 4.21k/4.21k [00:03<00:00, 1.34kB/s]


全部 2 个文件上传成功!
[SUCCESS] [Resources] idx=50 2025-05-15-015 study_1_2025-05-15-015 RT_Series9_2 /Users/kukudehui/Desktop/XNAT_DATA/sorted_data/2025-05-15-015/study_1_1.2.40.0.13.1.1.192.192.186.171.20250520070000051.32841/RT_RTRECORD/Series9_2


In [3]:
delete_subject(session, 'SYOncologySR', '2025-05-15-015')
delete_subject(session, 'SYOncologySR', '2025-09-24-019')
delete_subject(session, 'SYOncologySR', '2025-10-21-006')
delete_subject(session, 'SYOncologySR', '2025-11-03-010')
delete_subject(session, 'SYOncologySR', '2025-11-09-001')
delete_subject(session, 'SYOncologySR', '2025-12-05-002')

(404,
 '<html>\n<head>\n   <title>Status page</title>\n</head>\n<body>\n<h3>Unable to find the specified subject.</h3><p>You can get technical details <a href="http://www.w3.org/Protocols/rfc2616/rfc2616-sec10.html#sec10.4.5">here</a>.<br>\nPlease continue your visit at our <a href="/">home page</a>.\n</p>\n</body>\n</html>\n')